# Clase 10 — Monitorización

**Curso:** MLOps y Monitorización con Python · **Bloque B**
**Tiempo estimado:** 100 minutos · **Capítulo del manual:** `manual/10_monitorizacion.md`

## Qué verás en esta clase

- La diferencia entre **métricas de sistema** (latencia, tasa de error, uso de CPU/memoria) y **métricas de modelo** (distribución de las predicciones, un proxy de calidad cuando no hay etiqueta inmediata).
- Cómo instrumentar la API de churn de la clase 7 con **logging estructurado** real (el módulo `logging` de Python, con un formateador JSON), en vez de `print`.
- Cómo simular, con una semilla fija, un día completo de tráfico con **2 incidentes sintéticos** insertados a propósito (un pico de latencia, una ola de errores).
- Cómo construir un **panel** que resuma esas métricas por ventana horaria, con una tabla de `pandas` (la fuente verificable de cada cifra) y un gráfico de `matplotlib` guardado en disco.
- Cómo definir **alertas** basadas en umbrales que detecten los 2 incidentes reales sin falsos positivos, y el compromiso concreto entre alertar demasiado y demasiado poco.

Todo ocurre dentro de una carpeta de trabajo efímera (`proyecto_churn/`), creada al principio y **eliminada al final** de este mismo notebook. Ningún "tiempo de respuesta" de este notebook se mide con el reloj real: todos los tiempos son valores sintéticos, generados con una semilla fija, para que la clase sea idéntica en cada ejecución.

## Prerequisitos

- De la clase 7: la API FastAPI (`Cliente`, `/predecir`, `TestClient`), que este notebook reconstruye y amplía con logging.
- De las clases 4 y 5: la configuración ganadora del modelo (regresión logística, `class_weight="balanced"`, `C=1,0`).
- Python básico: funciones, diccionarios, el módulo `logging` de la biblioteca estándar (esta clase lo introduce desde cero).

> **Cómo usar este notebook:** ejecuta las celdas en orden, de arriba a abajo, una sola vez. Tarda menos de un minuto en total.

---
## Instalación

Instala solo lo que falte (FastAPI, httpx, matplotlib).

In [ ]:
import warnings

# Silenciamos un aviso de compatibilidad de starlette/httpx (fijo, no aporta nada aquí) para que
# la salida de este notebook no dependa de detalles internos de versiones de librerías
warnings.filterwarnings("ignore", category=DeprecationWarning)

import subprocess
import sys


def asegurar(paquete, modulo):
    """Instala el paquete solo si falta (idempotente y silencioso)."""
    try:
        __import__(modulo)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", paquete], check=True)

asegurar("numpy", "numpy")
asegurar("pandas", "pandas")
asegurar("scikit-learn", "sklearn")
asegurar("matplotlib", "matplotlib")
asegurar("joblib", "joblib")
asegurar("fastapi", "fastapi")
asegurar("httpx", "httpx")
print("Librerías listas.")

---
## Celda 0 — Setup

In [ ]:
import os
import shutil
import json
import logging
import warnings

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import joblib
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, roc_auc_score

# Silenciamos un aviso de compatibilidad de starlette/httpx (fijo, no aporta nada aquí) para que
# la salida de este notebook no dependa de detalles internos de versiones de librerías
warnings.filterwarnings("ignore", category=DeprecationWarning)

def cargar_datos(nombre):
    """Busca un archivo de datos en datos/, ../datos/ y /content/datos/."""
    for base in ("datos", "../datos", "/content/datos"):
        ruta = os.path.join(base, nombre)
        if os.path.exists(ruta):
            return ruta
    # Copia pubblica (Colab): se non c'è in locale, scarica il file dal repository dei notebook.
    try:
        import urllib.request
        destino = os.path.join("datos", nombre)
        os.makedirs(os.path.dirname(destino), exist_ok=True)
        urllib.request.urlretrieve(f"https://raw.githubusercontent.com/RaffaeleBini/arcadia-notebooks/main/18-mlops-monitoraggio/datos/{nombre}", destino)
        return destino
    except Exception:
        pass
    raise FileNotFoundError(
        "No encuentro " + nombre + ". Sube la carpeta datos/ del curso (en Colab: panel de archivos) o déjala junto al notebook.")

SEMILLA = 42
COLUMNAS_CATEGORICAS = ["plan", "contrato", "metodo_pago"]
COLUMNAS_NUMERICAS = ["antiguedad_meses", "uso_mensual_horas", "tickets_soporte", "factura_mensual"]
COLUMNAS_ENTRADA = COLUMNAS_CATEGORICAS + COLUMNAS_NUMERICAS

print("Librerías y funciones auxiliares listas.")

---
## Carpeta de trabajo efímera (se limpia al principio y al final)

In [ ]:
CARPETA = "proyecto_churn"

# Limpiamos y recreamos la carpeta para que el notebook sea repetible desde cero (también al empezar, por si quedó algo de una ejecución anterior)
if os.path.exists(CARPETA):
    shutil.rmtree(CARPETA)
os.makedirs(os.path.join(CARPETA, "logs"))
print("Carpeta de trabajo creada:", os.path.abspath(CARPETA))

---
## 1. Reentrenar el modelo y reconstruir la API de la clase 7

Esta clase no depende de ningún archivo guardado en una clase anterior: reentrena la misma Pipeline de las clases 4, 5 y 7, y reconstruye un único endpoint (`/predecir`) de la API FastAPI de la clase 7 -- suficiente para generar tráfico que monitorizar.

In [ ]:
RUTA_CSV = cargar_datos("churn.csv")
df = pd.read_csv(RUTA_CSV)

X = df[COLUMNAS_ENTRADA]
y = df["abandono"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=SEMILLA, stratify=y
)

# Misma configuración ganadora de las clases 4, 5 y 7: regresión logística, class_weight="balanced", C=1.0
preprocesador = ColumnTransformer([
    ("categoricas", OneHotEncoder(handle_unknown="ignore"), COLUMNAS_CATEGORICAS),
    ("numericas", StandardScaler(), COLUMNAS_NUMERICAS),
])
clasificador = LogisticRegression(class_weight="balanced", random_state=SEMILLA, max_iter=1000, C=1.0)
pipeline = Pipeline([("preprocesado", preprocesador), ("clasificador", clasificador)])
pipeline.fit(X_train, y_train)

f1 = round(f1_score(y_test, pipeline.predict(X_test)), 4)
auc = round(roc_auc_score(y_test, pipeline.predict_proba(X_test)[:, 1]), 4)
print("F1 (conjunto de prueba):", f1)
print("AUC (conjunto de prueba):", auc)
assert f1 > 0.25 and auc > 0.6, "el reentrenamiento no reprodujo el modelo esperado"

RUTA_MODELO = os.path.join(CARPETA, "modelo.pkl")
joblib.dump(pipeline, RUTA_MODELO)
MODELO = joblib.load(RUTA_MODELO)
assert np.array_equal(MODELO.predict(X_test), pipeline.predict(X_test))
print("Modelo reentrenado (igual que en la clase 7) y guardado como artefacto en", RUTA_MODELO)

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal
from fastapi import FastAPI
from fastapi.testclient import TestClient


class Cliente(BaseModel):
    """El mismo modelo Pydantic de la clase 7: las columnas de entrada de churn.csv."""
    id_cliente: str
    antiguedad_meses: int = Field(ge=0)
    plan: Literal["basico", "estandar", "premium"]
    contrato: Literal["mensual", "anual"]
    uso_mensual_horas: float = Field(ge=0)
    tickets_soporte: int = Field(ge=0)
    metodo_pago: Literal["domiciliacion", "tarjeta", "transferencia"]
    factura_mensual: float = Field(ge=0)


def cliente_a_fila(cliente):
    datos = cliente.model_dump()
    return pd.DataFrame([datos])[COLUMNAS_ENTRADA]


app = FastAPI(title="API de churn (clase 7, ampliada con monitorización en esta clase)")


@app.post("/predecir")
def predecir(cliente: Cliente):
    fila = cliente_a_fila(cliente)
    probabilidad = float(MODELO.predict_proba(fila)[0, 1])
    return {
        "id_cliente": cliente.id_cliente,
        "probabilidad_abandono": round(probabilidad, 4),
        "abandono_previsto": int(probabilidad >= 0.5),
    }


cliente_api = TestClient(app)
print("API de la clase 7 reconstruida (un solo endpoint, /predecir, es suficiente para esta clase). Lista para instrumentar con logging.")

---
## 2. Logging estructurado: un formateador JSON real

En vez de `print`, definimos un `logging.Formatter` que convierte cada evento en una línea JSON con 7 campos siempre presentes (`timestamp`, `nivel`, `endpoint`, `id_cliente`, `status_code`, `latencia_ms`, `probabilidad_abandono`). El campo `timestamp` es un valor que NOSOTROS pasamos explícitamente (con `extra=...`), no la hora real que `logging` capturaría por defecto -- así el log entero es reproducible.

In [ ]:
class FormateadorJSON(logging.Formatter):
    """Formatea cada LogRecord como una línea JSON con campos consistentes, en vez de un mensaje de texto libre.

    El campo "timestamp" viene de un valor SINTÉTICO que nosotros pasamos con extra=..., no del reloj real del sistema (record.created): así el log entero es reproducible con una semilla fija, algo que no podríamos garantizar con la hora real de ejecución.
    """

    def format(self, record):
        registro = {
            "timestamp": getattr(record, "timestamp", None),
            "nivel": record.levelname,
            "endpoint": getattr(record, "endpoint", None),
            "id_cliente": getattr(record, "id_cliente", None),
            "status_code": getattr(record, "status_code", None),
            "latencia_ms": getattr(record, "latencia_ms", None),
            "probabilidad_abandono": getattr(record, "probabilidad_abandono", None),
            "mensaje": record.getMessage(),
        }
        return json.dumps(registro, ensure_ascii=False)


RUTA_LOG = os.path.join(CARPETA, "logs", "peticiones.log")

logger_api = logging.getLogger("api_churn")
logger_api.setLevel(logging.INFO)
logger_api.handlers.clear()  # evita duplicar handlers si esta celda se ejecuta más de una vez
manejador = logging.FileHandler(RUTA_LOG, mode="w", encoding="utf-8")
manejador.setFormatter(FormateadorJSON())
logger_api.addHandler(manejador)
logger_api.propagate = False


def registrar_peticion(timestamp, endpoint, id_cliente, status_code, latencia_ms, probabilidad_abandono=None):
    """Registra una petición con logging ESTRUCTURADO real (no print): cada llamada añade una línea JSON al archivo de log, con los mismos campos siempre en el mismo lugar."""
    logger_api.info(
        "peticion",
        extra={
            "timestamp": timestamp, "endpoint": endpoint, "id_cliente": id_cliente,
            "status_code": status_code, "latencia_ms": round(latencia_ms, 3),
            "probabilidad_abandono": None if probabilidad_abandono is None else round(probabilidad_abandono, 4),
        },
    )


print("Logger estructurado listo. Cada línea escrita en", RUTA_LOG, "será JSON válido, con los mismos 7 campos siempre.")

---
## 3. Una demostración con la API real

Antes de simular un día entero, tres peticiones reales a través de `TestClient`, registradas con `registrar_peticion`. Las "latencias" de esta demostración son valores fijos de ejemplo (no medidos con el reloj): esta clase distingue explícitamente "probar que la API responde" (aquí) de "medir cuánto tarda" (clase 7, con la excepción declarada de esa clase; en esta clase 10 no medimos tiempo real en ningún momento).

In [ ]:
# Tres peticiones de ejemplo, con latencias SINTÉTICAS fijas (no medidas con time.perf_counter): en esta clase
# NINGÚN "tiempo de respuesta" se mide con el reloj real -- todos, incluida esta demostración, son valores de ejemplo predefinidos, para que el notebook sea idéntico en cada ejecución
cliente_demo = {
    "id_cliente": "C_DEMO_0001", "antiguedad_meses": 12, "plan": "estandar", "contrato": "mensual",
    "uso_mensual_horas": 15.0, "tickets_soporte": 1, "metodo_pago": "tarjeta", "factura_mensual": 22.5,
}

respuesta_ok = cliente_api.post("/predecir", json=cliente_demo)
registrar_peticion("2026-01-12T09:00:00", "/predecir", cliente_demo["id_cliente"], respuesta_ok.status_code, 12.3,
                    respuesta_ok.json()["probabilidad_abandono"])

cliente_invalido = dict(cliente_demo, plan="diamante")
respuesta_422 = cliente_api.post("/predecir", json=cliente_invalido)
registrar_peticion("2026-01-12T09:00:01", "/predecir", cliente_invalido["id_cliente"], respuesta_422.status_code, 3.1)

respuesta_lenta = cliente_api.post("/predecir", json=cliente_demo)
registrar_peticion("2026-01-12T09:00:02", "/predecir", cliente_demo["id_cliente"], respuesta_lenta.status_code, 181.4,
                    respuesta_lenta.json()["probabilidad_abandono"])

with open(RUTA_LOG, encoding="utf-8") as f:
    lineas_demo = f.readlines()
print("Líneas escritas hasta ahora:", len(lineas_demo))
for linea in lineas_demo:
    print(linea.strip())
    json.loads(linea)  # confirma que cada línea es JSON válido

---
## 4. Simular un día de tráfico con 2 incidentes sintéticos

`generar_registro_dia` recorre 24 ventanas horarias y, para cada una, un número variable de peticiones (entre 30 y 70), todo con la misma `SEMILLA`. En la hora 14 la latencia se dispara (media ≈160 ms en vez de ≈12 ms); en la hora 20 la tasa de error se dispara (≈32 % en vez de ≈1,2 %). Cada petición se registra con `registrar_peticion`: el log resultante es la fuente de datos de todo el resto de la clase.

> **Atención:** ninguna latencia de esta sección se mide con `time.perf_counter()` ni con ningún reloj real. Son valores generados con `numpy.random.default_rng(SEMILLA)`, exactamente reproducibles.

In [ ]:
# Ventanas horarias con incidente sintético insertado a propósito -- NO se eligen al azar en cada ejecución,
# son constantes fijas del ejercicio, exactamente como pide el enunciado de la clase 10
HORA_INCIDENTE_LATENCIA = 14   # pico de latencia entre las 14:00 y las 15:00
HORA_INCIDENTE_ERRORES = 20    # ola de errores entre las 20:00 y las 21:00

LATENCIA_BASE_MEDIA_MS, LATENCIA_BASE_DESV_MS = 12.0, 2.5
LATENCIA_INCIDENTE_MEDIA_MS, LATENCIA_INCIDENTE_DESV_MS = 160.0, 30.0
TASA_ERROR_BASE = 0.012
TASA_ERROR_INCIDENTE = 0.32
FECHA_BASE = "2026-01-12"

rng = np.random.default_rng(SEMILLA)


def generar_registro_dia():
    """Genera, con la semilla fija SEMILLA, un día completo (24 ventanas horarias) de peticiones simuladas a /predecir, con 2 incidentes sintéticos insertados a propósito. Cada petición se registra con logging estructurado real (registrar_peticion), no con print."""
    total = 0
    for hora in range(24):
        n_peticiones = int(rng.integers(30, 71))
        for i in range(n_peticiones):
            idx = rng.integers(0, len(df))
            fila = df.iloc[[idx]][COLUMNAS_ENTRADA]
            probabilidad = float(MODELO.predict_proba(fila)[0, 1])

            if hora == HORA_INCIDENTE_LATENCIA:
                latencia = float(rng.normal(LATENCIA_INCIDENTE_MEDIA_MS, LATENCIA_INCIDENTE_DESV_MS))
            else:
                latencia = float(rng.normal(LATENCIA_BASE_MEDIA_MS, LATENCIA_BASE_DESV_MS))
            latencia = max(latencia, 0.1)  # una latencia no puede ser negativa

            p_error = TASA_ERROR_INCIDENTE if hora == HORA_INCIDENTE_ERRORES else TASA_ERROR_BASE
            es_error = rng.random() < p_error
            status_code = 500 if es_error else 200

            minuto = int(i * 60 / n_peticiones)
            timestamp = f"{FECHA_BASE}T{hora:02d}:{minuto:02d}:00"
            id_cliente = f"C_SIM_{hora:02d}_{i:04d}"
            registrar_peticion(timestamp, "/predecir", id_cliente, status_code, latencia, probabilidad)
            total += 1
    return total


N_REGISTRADOS = generar_registro_dia()
print("Peticiones simuladas y registradas en el log:", N_REGISTRADOS)
assert N_REGISTRADOS > 3 * 24, "esperábamos varias decenas de peticiones por hora"

---
## 5. Leer el log estructurado (como haría un sistema real de análisis)

In [ ]:
registros = []
with open(RUTA_LOG, encoding="utf-8") as f:
    for linea in f:
        registros.append(json.loads(linea))

df_log = pd.DataFrame(registros)
df_log["hora"] = df_log["timestamp"].str.slice(11, 13).astype(int)
print("Filas totales en el log (incluye las 3 peticiones de demostración):", len(df_log))
print(df_log.head(3).to_string(index=False))

---
## 6. El panel: una tabla agregada por ventana horaria

`panel` es una tabla de `pandas` con una fila por hora: número de peticiones, tasa de error, latencia media, latencia p95 y la probabilidad media prevista (una **métrica de modelo**: un proxy de calidad, útil cuando -- como en producción -- no tenemos la etiqueta real `abandono` de un cliente en el momento de predecir).

In [ ]:
# Panel: una fila por ventana horaria, con las métricas de SISTEMA (peticiones, tasa de error, latencia)
# y una métrica de MODELO (probabilidad media prevista): un proxy de calidad que no necesita conocer la etiqueta real
df_peticiones_simuladas = df_log[df_log["id_cliente"].str.startswith("C_SIM_")].copy()

panel = (
    df_peticiones_simuladas.groupby("hora")
    .agg(
        n_peticiones=("status_code", "size"),
        tasa_error=("status_code", lambda s: (s >= 500).mean()),
        latencia_media_ms=("latencia_ms", "mean"),
        latencia_p95_ms=("latencia_ms", lambda s: s.quantile(0.95)),
        probabilidad_media=("probabilidad_abandono", "mean"),
    )
    .reset_index()
)
pd.set_option("display.width", 140)
print(panel.round(4).to_string(index=False))
assert len(panel) == 24, "el panel debe tener exactamente una fila por cada una de las 24 horas"

**¿Qué observas?** Las horas 14 y 20 destacan claramente en la tabla: la hora 14 tiene una `latencia_media_ms` casi 13 veces mayor que las horas normales, y la hora 20 tiene una `tasa_error` unas 27 veces mayor. La `probabilidad_media` (métrica de modelo), en cambio, se mantiene en un rango parecido en las 24 horas -- en este log no simulamos ningún cambio en la distribución de las predicciones, solo incidentes de sistema. Eso es exactamente lo que la clase 11 (*drift*) investigará: qué hacer cuando SÍ cambia esa distribución.

---
## 7. El mismo panel, como gráfico (guardado, no mostrado)

In [ ]:
# Guardamos el gráfico en disco pero NO lo mostramos (plt.show()): el texto de la tabla de arriba
# es la fuente verificable de cada número; el gráfico es solo un apoyo visual complementario
figura, (ax_latencia, ax_error) = plt.subplots(2, 1, figsize=(9, 6), sharex=True)

ax_latencia.plot(panel["hora"], panel["latencia_media_ms"], marker="o", color="tab:blue")
ax_latencia.set_ylabel("Latencia media (ms)")
ax_latencia.set_title("Métricas de sistema por ventana horaria (día simulado)")
ax_latencia.grid(alpha=0.3)

ax_error.plot(panel["hora"], panel["tasa_error"] * 100, marker="o", color="tab:red")
ax_error.set_ylabel("Tasa de error (%)")
ax_error.set_xlabel("Hora del día")
ax_error.grid(alpha=0.3)

RUTA_GRAFICO = os.path.join(CARPETA, "panel_monitorizacion.png")
figura.tight_layout()
figura.savefig(RUTA_GRAFICO, dpi=100)
plt.close(figura)
print("Gráfico guardado en:", RUTA_GRAFICO)
assert os.path.exists(RUTA_GRAFICO)

---
## 8. Alertas basadas en umbrales

`detectar_incidentes` marca una ventana horaria si su latencia media o su tasa de error superan un umbral. Con los umbrales de producción elegidos (50 ms, 10 %), detecta EXACTAMENTE las horas 14 y 20 -- ni una hora normal más, ni una menos.

In [ ]:
def detectar_incidentes(panel_agregado, umbral_latencia_ms, umbral_tasa_error):
    """Devuelve la lista ORDENADA de horas cuya latencia media o cuya tasa de error superan el umbral dado. Es la definición completa de una alerta basada en umbrales."""
    supera = (
        (panel_agregado["latencia_media_ms"] > umbral_latencia_ms)
        | (panel_agregado["tasa_error"] > umbral_tasa_error)
    )
    return sorted(panel_agregado.loc[supera, "hora"].tolist())


# Umbrales de producción: se eligen mirando el panel -- muy por encima del comportamiento normal
# (latencia media normal ≈12 ms, tasa de error normal ≈0,9 %) y muy por debajo de los dos incidentes reales
# (≈154 ms de latencia, ≈24 % de errores): un margen amplio a ambos lados
UMBRAL_LATENCIA_MS = 50.0
UMBRAL_TASA_ERROR = 0.10

incidentes_reales = {HORA_INCIDENTE_LATENCIA, HORA_INCIDENTE_ERRORES}
incidentes_detectados = detectar_incidentes(panel, UMBRAL_LATENCIA_MS, UMBRAL_TASA_ERROR)
print("Horas marcadas por la alerta con los umbrales de producción:", incidentes_detectados)

# Comprobación concreta: detecta EXACTAMENTE los 2 incidentes insertados, sin falsos positivos en las 22 horas normales
assert set(incidentes_detectados) == incidentes_reales, "la alerta debía marcar exactamente las horas 14 y 20"
assert len(incidentes_detectados) == len(incidentes_reales), "no debía haber falsos positivos con este umbral"
print("Los 2 incidentes sintéticos se detectan correctamente y no hay ningún falso positivo entre las 22 horas normales.")

---
## 9. El compromiso: alertar demasiado frente a alertar demasiado poco

Sobre este mismo log, comprobamos los dos extremos del compromiso con números concretos: un umbral demasiado agresivo genera falsos positivos (fatiga de alertas); un umbral demasiado alto pierde los incidentes reales.

In [ ]:
# Umbral demasiado AGRESIVO (demasiado bajo): "alertar demasiado" -- fatiga de alertas
UMBRAL_LATENCIA_AGRESIVO = 13.0
UMBRAL_TASA_ERROR_AGRESIVO = 0.02
incidentes_agresivo = detectar_incidentes(panel, UMBRAL_LATENCIA_AGRESIVO, UMBRAL_TASA_ERROR_AGRESIVO)
falsos_positivos_agresivo = sorted(set(incidentes_agresivo) - incidentes_reales)
print("Con un umbral agresivo (latencia > 13 ms o error > 2 %), horas marcadas:", incidentes_agresivo)
print("De ellas, falsos positivos (horas normales marcadas por error):", len(falsos_positivos_agresivo), falsos_positivos_agresivo)
assert len(falsos_positivos_agresivo) > 0, "el umbral agresivo debía producir al menos un falso positivo, para ilustrar la fatiga de alertas"

# Umbral demasiado ALTO: "alertar demasiado poco" -- los incidentes reales pasan desapercibidos
UMBRAL_LATENCIA_ALTO = 500.0
UMBRAL_TASA_ERROR_ALTO = 0.60
incidentes_umbral_alto = detectar_incidentes(panel, UMBRAL_LATENCIA_ALTO, UMBRAL_TASA_ERROR_ALTO)
incidentes_perdidos = sorted(incidentes_reales - set(incidentes_umbral_alto))
print("Con un umbral demasiado alto (latencia > 500 ms o error > 60 %), horas marcadas:", incidentes_umbral_alto)
print("De los 2 incidentes reales, cuántos se pierden por completo:", len(incidentes_perdidos), incidentes_perdidos)
assert len(incidentes_perdidos) == 2, "el umbral demasiado alto debía perder los 2 incidentes reales, para ilustrar el otro extremo del compromiso"

print()
print("Resumen del compromiso, con las cifras de este mismo log:")
print("-- umbral agresivo: 0 incidentes perdidos, pero", len(falsos_positivos_agresivo), "falsos positivos de 22 horas normales")
print("-- umbral de producción (elegido en esta clase): 0 falsos positivos y 0 incidentes perdidos")
print("-- umbral demasiado alto: 0 falsos positivos, pero", len(incidentes_perdidos), "incidentes reales perdidos de 2")

**¿Qué observas?** Con las cifras recalculadas en este mismo notebook: el umbral agresivo (13 ms / 2 %) marca varias horas normales como incidentes -- puro ruido estadístico de la variación hora a hora, no un problema real -- mientras que el umbral de producción (50 ms / 10 %) no comete ese error porque deja un margen amplio sobre la variación normal. El umbral demasiado alto (500 ms / 60 %), en el otro extremo, no marca NADA: los 2 incidentes reales de este log pasarían desapercibidos para cualquiera que solo mirara ese panel de alertas. Ningún umbral es "el correcto" en abstracto: depende de cuánto ruido de fondo tenga tu sistema y de cuánto cueste (en tiempo de alguien) revisar una alerta falsa frente a lo que cuesta no ver un incidente real.

---
## 10. Resumen y cierre

- Distinguimos métricas de **sistema** (latencia, tasa de error) de métricas de **modelo** (distribución de las predicciones, un proxy sin etiqueta inmediata).
- Instrumentamos la API de la clase 7 con **logging estructurado** real (`logging` + un formateador JSON), no `print`.
- Simulamos, con semilla fija, un día de tráfico con 2 incidentes sintéticos, y construimos un panel (tabla + gráfico guardado) que los hace visibles.
- Definimos alertas por umbral que detectan los 2 incidentes reales sin falsos positivos, y medimos con números concretos el compromiso entre umbrales demasiado agresivos y demasiado altos.

*Continúa con la clase 11: Drift y reentrenamiento.*

---
## Limpieza final

In [ ]:
shutil.rmtree(CARPETA)
print("Carpeta de trabajo eliminada:", CARPETA)
assert not os.path.exists(CARPETA)